# Tutorial: Calibration and Patch Lifecycle

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Model calibrators and maintainers responsible for baseline updates.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Run calibration against configured objectives.
- Promote latest patch into run config.
- Restore previous baseline snapshot when needed.


## Outline

1. Inspect calibration spec
2. Run calibration optimizer
3. Inspect generated patch artifacts
4. Promote patch and verify
5. Rollback from snapshot if needed


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: Inspect calibration spec


In [ ]:
spec = REPO / "configs" / "calibration.yml"
print("Calibration spec:", spec)
if spec.exists():
    print(spec.read_text(encoding="utf-8")[:1200])


## Step 2: Run calibration (heavy)


In [ ]:
if RUN_CALIBRATION:
    _ = sh(f"python scripts/calibration/calibrate_model.py --config {CONFIG} --calibration-spec configs/calibration.yml", cwd=REPO)
else:
    print("Set RUN_CALIBRATION=True to execute calibration")


## Step 3: Find latest patch


In [ ]:
patches = sorted((REPO / "outputs" / "runs" / "calibration").glob("**/best_config_patch.yml"))
print("patches found:", len(patches))
latest_patch = patches[-1] if patches else None
print("latest patch:", latest_patch)
if latest_patch:
    print(latest_patch.read_text(encoding="utf-8")[:1200])


## Step 4: Promote patch into config


In [ ]:
if RUN_CALIBRATION and latest_patch is not None:
    _ = sh(f"python scripts/calibration/calibration_cycle.py promote --config {CONFIG} --patch {latest_patch}", cwd=REPO)
else:
    print("Promotion skipped: enable RUN_CALIBRATION and ensure latest_patch exists")


## Step 5: Inspect restore snapshots


In [ ]:
snaps = sorted((REPO / "outputs" / "runs" / "calibration" / "cycle").glob("**/baseline_before.yml"))
print("snapshot count:", len(snaps))
if snaps:
    print("latest snapshot:", snaps[-1])


## Step 6: Rollback command (manual gate)


In [ ]:
if snaps:
    cmd = f"python scripts/calibration/calibration_cycle.py restore --config {CONFIG} --snapshot {snaps[-1]}"
    print(cmd)


## Pitfalls

- Promote only after comparing baseline-vs-calibrated outputs.
- Store calibration run metadata and random seed for reproducibility.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
